# Áreas Protegidas — Sistema Nacional de Áreas Protegidas (SINAP) y SMByC

## Qué es el SINAP y por qué Colombia lo monitorea estadísticamente

Colombia es uno de los países más biodiversos del mundo — megadiverso por decreto geográfico, con dos océanos, la Amazonia, los Andes y el Pacífico húmedo convergiendo en su territorio. El **Sistema Nacional de Áreas Protegidas (SINAP)** es el instrumento principal para proteger este patrimonio: incluye Parques Nacionales Naturales, reservas forestales, distritos de manejo integrado y áreas protegidas privadas, sumando más de **31 millones de hectáreas** (alrededor del 27 % del territorio continental).

El monitoreo cuantitativo del SINAP se centra en dos preguntas de alto impacto político y científico: ¿cuánta cobertura boscosa se está perdiendo dentro y fuera de las áreas protegidas? y ¿las áreas protegidas son efectivas para frenar la deforestación?. El **Sistema de Monitoreo de Bosques y Carbono (SMByC)** del IDEAM genera las cifras oficiales de deforestación anuales que alimentan los compromisos de la NDC colombiana y los reportes ante el Convenio de Diversidad Biológica.

La **tasa anual de deforestación** es la variable clave de esta línea: Colombia perdió aproximadamente **171.685 hectáreas** de bosque en 2020 según el IDEAM — la Amazonia concentra más del 60 % de esta pérdida. El análisis estadístico de esta serie permite detectar si las políticas de conservación están funcionando, identificar departamentos y municipios de alto riesgo, y proyectar el estado futuro del bosque.

## Preguntas que este notebook responde

- ¿Existe una tendencia estadísticamente significativa (Mann-Kendall) en la tasa de deforestación en el período analizado?
- ¿Los modelos RandomForest o XGBoost pueden predecir la pérdida de cobertura con suficiente precisión para alertas tempranas?
- ¿Qué relación tienen las variables socioeconómicas (vías, demografía) con la intensidad de la deforestación?

## Instituciones clave

| Institución | Rol principal |
|---|---|
| Parques Nacionales Naturales (PNN) | Gestión de la red de áreas protegidas |
| IDEAM — SMByC | Monitoreo oficial de bosques y carbono, cifras de deforestación |
| Instituto Humboldt (IAvH) | Investigación en biodiversidad, representatividad ecológica |
| MADS | Política del SINAP, CONPES 4050 |
| IGAC | Cartografía base, catastro |

> **Contexto de dominio completo:** [`docs/fuentes/areas_protegidas.md`](../../docs/fuentes/areas_protegidas.md)  
> **Bloque:** A — Gestión | **Línea:** `areas_protegidas`  
> **Variable principal:** `cobertura_ha` (hectáreas de bosque) / `tasa_deforestacion` (%/año)  
> **Modelos sugeridos:** RANDOM_FOREST, XGBOOST (series anuales — sin componente estacional mensual)  
> **Mann-Kendall:** obligatorio — es el único test que evidencia tendencias en series anuales cortas  
> **Sin ENSO lag:** la deforestación responde a motores socioeconómicos, no directamente al ENSO

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "areas_protegidas"
VARIABLE = "cobertura_ha"
UNIDAD = "ha"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `areas_protegidas` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "areas_protegidas.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

La fuente oficial para datos de cobertura boscosa y deforestación en Colombia es el **SMByC del IDEAM**, que publica cifras anuales desagregadas por departamento, municipio, área protegida y bioma. Los datos están disponibles en el portal del IDEAM y a través de Global Forest Watch (GFW).

**Estructura típica de los datos SMByC:**

| Variable | Unidad | Descripción | Fuente |
|---|---|---|---|
| `cobertura_ha` | Hectáreas | Área de bosque natural al final del período | IDEAM SMByC |
| `perdida_ha` | Hectáreas | Pérdida bruta de bosque en el año | IDEAM SMByC |
| `tasa_deforestacion` | %/año | `perdida_ha / cobertura_ha_inicial * 100` | Calculado |
| `efectividad_manejo` | % (0-100) | Puntaje EMAP o AEMAPPS del área protegida | PNN / CARs |
| `area_protegida` | Hectáreas | Extensión total del AP | RUNAP |

**Particularidad de las series anuales:** A diferencia de las otras líneas del proyecto (temperatura mensual, nivel diario), los datos del SMByC son **series anuales**. Esto implica que no hay estacionalidad mensual que modelar — SARIMA sin componente estacional (p,d,q)(0,0,0,0) o directamente RandomForest/XGBoost con covariables socioeconómicas.

**Normativa aplicable:**
- **Decreto 1076 de 2015** (compila Decreto 2372/2010): define el SINAP, las categorías de manejo y los objetivos de conservación
- **CONPES 4050 de 2021:** metas al 2030 para el SINAP — incluye deforestación cero en áreas protegidas
- **Ley 165 de 1994:** Convenio de Diversidad Biológica — obliga a reportar estado de los ecosistemas

> **Ruta esperada para datos reales:** `data/raw/areas_protegidas.csv`  
> Columnas mínimas: `fecha` (año, formato `YYYY-01-01`), `cobertura_ha`.  
> Columnas recomendadas: `perdida_ha`, `tasa_deforestacion`, `departamento`, `nombre_ap`.

In [ ]:
# df = load_csv("data/raw/areas_protegidas.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "cobertura_ha": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

La validación de datos de cobertura boscosa requiere atención especial a la consistencia entre períodos:

**Por qué `validate()` con `linea_tematica="areas_protegidas"` (ADR-006):** El validador verifica que la cobertura no aumente abruptamente de un año a otro sin explicación (podría indicar cambio metodológico en el SMByC, reclasificación de coberturas, o error en la fuente). Un incremento de > 5 % en un año generalmente no es biológicamente posible sin plantaciones masivas o recuperación de bosque secundario bien documentada.

**Problemas frecuentes en datos del SMByC:**

1. **Cambios metodológicos entre períodos:** El SMByC ha actualizado su metodología (Landsat 8, cambios en definición de bosque). Series largas pueden tener quiebres metodológicos que no son cambios reales en la cobertura.

2. **Subestimación por nubosidad:** Los mapas de cobertura tienen áreas "sin información" enmascaradas por nubes. La deforestación real puede ser mayor que la reportada en años muy nublados (efecto recurrente en Amazonia y Pacífico).

3. **Escala nacional vs. local:** Los datos nacionales del SMByC no tienen la misma precisión que los levantamientos específicos de un área protegida. Para un PNN específico, usar los datos del sistema PNNIS de Parques Nacionales.

El reporte EDA debe incluir análisis por departamento si los datos tienen esa desagregación. La varianza entre departamentos es enorme — Amazonas vs. Atlántico no son comparables.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_areas_protegidas.html",
       title="EDA — Áreas Protegidas", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

Las series anuales de cobertura boscosa o deforestación tienen dinámicas muy distintas a las series mensuales de temperatura o nivel:

- **No hay ciclo estacional:** la cobertura boscosa es una variable de estado que cambia anualmente. La visualización debe buscar **tendencia de largo plazo** y **anomalías por año** (picos de deforestación en años post-conflicto, post-acuerdo de paz, etc.).
- **El post-acuerdo de paz (2016-2017):** la firma del Acuerdo con las FARC redujo el control territorial en zonas de conflicto, provocando un aumento dramático de la deforestación en la Amazonia colombiana entre 2016 y 2019 — es el punto de quiebre más importante en las series recientes.
- **Variaciones interanuales explicadas:** años de El Niño intenso pueden coincidir con mayor deforestación (incendios en épocas de sequía), pero la relación causal principal es socioeconómica, no climática.
- **`plot_seasonal_means` con `period="year"`:** para series anuales, usar el promedio por año para identificar años atípicos. Una barra marcadamente mayor indica un año de deforestación acelerada.

En las visualizaciones de cobertura, usar el eje Y empezando cerca del rango de los datos (no desde cero) para que las variaciones sean visibles — la cobertura nacional rara vez baja a cero, pero las pérdidas anuales son significativas en términos relativos.

In [ ]:
plot_series(df, "fecha", "cobertura_ha", title="Áreas Protegidas — cobertura_ha (ha)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "cobertura_ha", period="month")
plt.show()

## 4. Estadística descriptiva

En series de cobertura boscosa, los estadísticos descriptivos tienen interpretaciones de política:

- **Media de cobertura (ha):** indica el nivel promedio de bosque en el período. Comparar con la línea base del NREF (Nivel de Referencia de Emisiones Forestales) para evaluar si el área está por encima o debajo del nivel histórico comprometido en la NDC.
- **Tasa de cambio media:** la pendiente promedio de pérdida anual. Una pérdida de 100.000 ha/año en un área de 5 millones de ha implica una tasa del 2 % anual — muy por encima de la meta de deforestación neta cero de la Agenda 2030.
- **Desviación estándar de la pérdida anual:** alta SD indica alta variabilidad entre años — la deforestación responde fuertemente a shocks políticos y socioeconómicos (acuerdos de paz, precios de commodities agrícolas, presencia militar).
- **Mínimo y máximo:** identificar el año de mayor pérdida (año crítico para el análisis político) y el de menor pérdida (mejor año de referencia para evaluar el efecto de políticas de conservación).

**Indicador oficial SINAP — ProtConn:** mide la conectividad de las áreas protegidas. No es calculable directamente desde una serie temporal de cobertura, pero la pérdida de bosque en corredores entre APs impacta directamente este indicador.

In [ ]:
summarize(df)

## 5. Análisis inferencial — Estacionariedad y Mann-Kendall en series anuales

### 5a. Prueba de estacionariedad (ADR-004) en series anuales

Para series anuales de cobertura boscosa, la estacionariedad tiene una interpretación directa: si la serie es estacionaria, el área de bosque oscila alrededor de un nivel medio estable — no hay pérdida neta sistemática. Si no es estacionaria (tendencia), el bosque está disminuyendo o aumentando de manera sostenida.

**Consideración especial con series cortas:** El SMByC produce datos desde 1990 con datos anuales consistentes desde ~2000. Una serie de 20-25 años tiene baja potencia estadística en ADF y KPSS. Interpretarlos con cautela: el valor-p puede ser no significativo no porque la tendencia no exista, sino porque la muestra es pequeña. Por eso Mann-Kendall complementa.

### 5b. Mann-Kendall — obligatorio para series anuales de deforestación

**Por qué Mann-Kendall es el test central en áreas protegidas:** Las series anuales de cobertura son cortas (20-25 puntos). El análisis de regresión lineal simple requeriría normalidad de residuos y linealidad estricta — condiciones que raramente se cumplen en series de deforestación con quiebres estructurales (post-conflicto, post-acuerdo). Mann-Kendall:
- No requiere normalidad
- Es robusto ante quiebres estructurales si se aplica correctamente
- Es el estándar metodológico de la FAO, el IPCC y el IDEAM para tendencias en coberturas forestales

**Interpretación aplicada al SINAP:**
- `trend = "decreasing"` con `p < 0.05` sobre `cobertura_ha`: pérdida de bosque estadísticamente confirmada — reportar en `docs/decisiones.md` y escalar al reporte de efectividad de manejo del AP.
- `slope` de Sen: pérdida anual promedio en ha/año. Multiplicar por el precio del carbono (≈ USD 5-15 por tCO₂e en mercados voluntarios) para estimar el valor económico de la deforestación.
- **Prueba de Pettitt:** complementa Mann-Kendall detectando el año exacto del quiebre. Útil para identificar el año en que una política de conservación tuvo efecto (o falló).

In [ ]:
ts = df.set_index("fecha")["cobertura_ha"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5c. Random Forest — Prediccion de deforestacion en areas protegidas

El **Random Forest** es el modelo de referencia del IDEAM/SMByC para predecir tasas de deforestacion. Ventajas frente a modelos lineales:

- Captura interacciones no lineales entre distancia a vias, pendiente, densidad demografica
- Robusto ante outliers y variables con diferentes escalas
- Produce importancia de variables (feature importance) para priorizar monitoreo

Variables predictoras tipicas del SMByC:
| Variable | Tipo | Fuente |
|---|---|---|
| Distancia a vias | Continua | IGAC/INVIAS |
| Pendiente | Continua | DEM NASA/IGAC |
| Densidad poblacional | Continua | DANE |
| Cobertura previa | Categorica | Corine Land Cover |
| Precipitacion media | Continua | IDEAM |
| Presencia mineria | Binaria | ANM/ANLA |

> **Matriz de confusion ajustada por area:** el SMByC usa exactitudes del usuario y productor ponderadas por el area real de cada clase (no por pixeles), siguiendo Olofsson et al. (2014) — estandar IPCC Tier 3.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)
N = 500  # puntos de muestreo en el area protegida

# Features: predictores de deforestacion
dist_vias_km  = np.random.exponential(15, N)           # km al vial mas cercano
pendiente_deg = np.random.gamma(2, 8, N).clip(0, 60)   # grados
dens_pob      = np.random.lognormal(2, 1, N)           # hab/km2
precipitacion = np.random.normal(2000, 400, N)          # mm/ano
mineria       = (np.random.random(N) < 0.15).astype(int)  # presencia mineria

X_rf = np.column_stack([dist_vias_km, pendiente_deg, dens_pob, precipitacion, mineria])
feature_names = ['dist_vias_km','pendiente_deg','dens_pob','precipitacion','mineria']

# Probabilidad de deforestacion (proceso generativo)
prob_def = 1 / (1 + np.exp(
    2 - 0.15*dist_vias_km + 0.02*dens_pob - 0.01*pendiente_deg + 1.5*mineria))
y_rf = (prob_def > 0.5).astype(int)  # 1=deforestado, 0=estable

# Entrenar RandomForest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
cv_scores = cross_val_score(rf, X_rf, y_rf, cv=5, scoring='f1')
rf.fit(X_rf, y_rf)
y_pred_rf = rf.predict(X_rf)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Panel A: Importancia de variables
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
ax1.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx],
         color=['#e74c3c' if i == sorted_idx[0] else '#3498db' for i in range(len(feature_names))],
         alpha=0.85)
ax1.set_title('Random Forest — Importancia de variables\n(prediccion deforestacion SMByC)', fontweight='bold')
ax1.set_xlabel('Importancia (Gini)'); ax1.grid(axis='x', alpha=0.3)

# Panel B: Matriz de confusion
cm = confusion_matrix(y_rf, y_pred_rf)
cm_norm = cm / cm.sum(axis=1, keepdims=True)  # normalizada por clase (exactitud productor)
im = ax2.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax2, label='Proporcion')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.2f})',
                 ha='center', va='center', fontsize=10)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred: Estable','Pred: Deforest.'])
ax2.set_yticklabels(['Real: Estable','Real: Deforest.'])
ax2.set_title('Matriz de confusion RF — Exactitud productor/usuario\n(Olofsson 2014 requiere ajuste por area)', fontweight='bold')

plt.tight_layout(); plt.show()
print(f'CV F1-score (5-fold): {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'Variable mas importante: {feature_names[importances.argmax()]} ({importances.max():.3f})')
print(classification_report(y_rf, y_pred_rf, target_names=['Estable','Deforestado']))

## 5b. Análisis de excedencias normativas en áreas protegidas

Para cobertura boscosa no existe una norma de cumplimiento con umbral numérico fijo como las normas de calidad del aire. Sin embargo, existen metas comprometidas internacionalmente que funcionan como umbrales de referencia:

**Metas de política como umbrales operativos:**

| Meta | Umbral | Fuente | Año objetivo |
|---|---|---|---|
| Deforestación neta cero en APs | Pérdida = 0 ha/año | CONPES 4050 de 2021 | 2030 |
| NDC Colombia — reducción deforestación | Reducción 51 % vs. 2014-2018 | NDC 2022 (MADS) | 2030 |
| Convenio Diversidad Biológica — 30x30 | 30 % territorio bajo protección | CDB Kunming-Montreal | 2030 |
| NREF nacional — línea base REDD+ | Pérdida ≤ promedio 2000-2012 | IDEAM SMByC | Anual |

Estos umbrales pueden configurarse en `exceedance_report()` de forma personalizada. Si la pérdida anual supera el nivel de referencia del NREF, hay un incumplimiento del compromiso REDD+ ante la CMNUCC.

**Para calidad del agua asociada al AP (si aplica):** Ver `config.NORMA_AGUA_POTABLE` y `config.NORMA_VERTIMIENTOS` para ríos que nacen dentro de las áreas protegidas (ADR-008).

In [ ]:
rep = exceedance_report(df["cobertura_ha"], variable="cobertura_ha")
if rep.empty:
    print("Sin normas colombianas registradas para 'cobertura_ha'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento — Manejo de series anuales con quiebres estructurales

Las series de cobertura boscosa colombianas tienen quiebres conocidos que no deben tratarse como datos faltantes ni como outliers:

**Quiebres estructurales documentados en la serie SMByC:**
- **2016-2019:** aumento acelerado de deforestación post-acuerdo de paz (reducción de control territorial en zonas FARC)
- **~2001-2002:** cambio metodológico en el SMByC (transición de Landsat 7 a metodología mejorada)
- **Años con alta nubosidad:** subestimación sistemática de deforestación

**Estrategia de imputación para series anuales:** Si hay un año sin dato (raro en el SMByC pero posible en datos de un AP específica), usar interpolación lineal entre el año anterior y el posterior — no la media histórica. Las series anuales de cobertura son monótonamente decrecientes en la mayoría de los casos; la media histórica sobreestimaría el valor faltante.

**ADR-002 aplicado a áreas protegidas:** Los años con alta deforestación (outliers hacia arriba en la pérdida) son exactamente los puntos más informativos para la política pública. No recortar ni suavizar estos valores. Si se aplica suavizado para visualización, reportarlo explícitamente como una operación estética, no analítica.

La imputación aplicada en esta sección se usa exclusivamente para garantizar que los modelos predictivos tengan series continuas. Los outliers preservados informan directamente la detección de quiebres y el test de Mann-Kendall.

In [ ]:
df_clean = impute(df.copy(), cols=["cobertura_ha"], method="linear")
print(f"Faltantes antes: {df["cobertura_ha"].isna().sum()} | "
      f"después: {df_clean["cobertura_ha"].isna().sum()}")

## 7. Modelos predictivos — RandomForest y XGBoost para series anuales sin estacionalidad

### Por qué NO usar SARIMA estacional para cobertura boscosa

Las series anuales de cobertura boscosa no tienen componente estacional mensual (el SMByC produce una cifra por año). SARIMA con componente estacional (P,D,Q,12) no tiene sentido — no hay 12 meses para modelar. En cambio, los modelos de Machine Learning son más apropiados porque permiten incorporar **covariables causales**:

- Distancia a vías (km) — motor principal de deforestación en Amazonia
- Densidad poblacional municipal (hab/km²)
- Precio de la coca (si el área es zona cocalera)
- Pendiente del terreno (DEM) — deforestación ocurre más en terrenos planos accesibles
- Presencia de cultivos de palma / ganadería (datos UPRA)
- Efectividad del manejo (puntaje EMAP del AP)

### RandomForest para cobertura boscosa

RandomForest captura interacciones no lineales entre estas covariables sin sobreajuste crítico — es el modelo estándar del SMByC y de Global Forest Watch para predicción de deforestación. Con lags temporales ([1,2,3] años) captura la dinámica de expansión agrícola que tiene inercia multiahual.

### XGBoost como alternativa

XGBoost suele superar a RandomForest cuando se dispone de suficientes covariables socioeconómicas estructuradas. Su ventaja es la capacidad de manejar datos heterogéneos y su velocidad de entrenamiento.

### Métricas de evaluación para series anuales

Usar **MAE y RMSE** en hectáreas — son directamente interpretables en términos de política. Un MAE de 5.000 ha/año significa que el modelo se equivoca en promedio en 5.000 ha — equivalente al área del Parque Nacional Tatamá. **No usar RMSLE** (la cobertura puede tomar valores muy grandes — ADR-003).

`walk_forward()` con `n_splits=3` (y no 4) es más apropiado para series de 20 puntos — evitar reducir excesivamente el conjunto de entrenamiento.

In [ ]:
ts = df_clean.set_index("fecha")["cobertura_ha"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

Al completar el análisis con datos reales, documentar aquí:

- **Tendencia Mann-Kendall:** `trend`, `p-value`, `slope Sen (ha/año)`. Interpretar respecto a las metas CONPES 4050 y NDC.
- **Año de mayor deforestación:** identificar y contextualizar (causa política/socioeconómica)
- **Modelo ganador:** nombre, MAE (ha/año), RMSE (ha/año), R² en walk-forward validation
- **Importancia de variables:** si se usaron covariables en RF/XGBoost, ¿cuál fue el predictor más importante?
- **Recomendaciones para gestión:** ¿requiere alerta temprana para algún AP específico?

### Normativa y referencias clave

| Instrumento | Aplicación en esta línea |
|---|---|
| Decreto 1076 de 2015 (compila Decreto 2372/2010) | Marco legal del SINAP y categorías de manejo |
| CONPES 4050 de 2021 | Política SINAP — metas a 2030 |
| Ley 165 de 1994 | Convenio sobre Diversidad Biológica |
| NREF — IDEAM SMByC | Línea base de deforestación para REDD+ |
| Resolución 1447 de 2018 (MADS) | Sistema MRV para reducción de emisiones |

---

## 9. Cómo adaptar a datos reales

**Paso 1 — Obtener datos del SMByC:**
- Portal IDEAM: http://smbyc.ideam.gov.co/
- Global Forest Watch (datos agregados con más filtros): https://www.globalforestwatch.org/
- Solicitud directa al SMByC para datos desagregados por área protegida o municipio

**Paso 2 — Preparar el archivo:**
```
data/raw/areas_protegidas.csv
```
Columnas mínimas requeridas:
```
fecha,cobertura_ha
2001-01-01,5820000
2002-01-01,5775000
...
```
Columnas recomendadas: `perdida_ha`, `tasa_deforestacion`, `departamento`, `nombre_ap`, `efectividad_manejo_pct`.

**Paso 3 — Ajustar frecuencia:**
Los datos son anuales. Asegurarse de que `pd.date_range(..., freq="YS")` se use en lugar de `freq="ME"`.

**Paso 4 — Agregar covariables socioeconómicas:**
Para mejorar RandomForest, cruzar con datos del IGAC (catastro), DANE (censo) o INVIAS (red vial) a nivel municipal o departamental.

**Paso 5 — Acceso al RUNAP para metadatos del AP:**
```python
# RUNAP — datos de áreas protegidas (descarga manual desde el portal)
# https://runap.parquesnacionales.gov.co/
# Columnas clave: nombre, categoria, departamento, area_ha, fecha_creacion
```

**Paso 6 — Para análisis espacial (recomendado):**
Usar Google Earth Engine o la plataforma SMByC para extraer datos de cobertura a nivel de polígono del AP específica. Exportar como CSV y cargar aquí.